# Step 4 of the PW: Future value of the firms (and Step 5 Magic Quadrant bridge)

| Question to answer (Presentation deck, Step 4) | How we measure it in this notebook |
|---|---|
| What does "future value" mean for Electrolux? | **Dual target:** (1) Top-10% cohort-normalized 5-year forward citations (breakthrough impact, Fleming 2001); (2) International multi-office expansion or US/EP grant (economic commitment) (Section 2) |
| Which universe trains and tests the model? | **Robotics wide universe** (`B25J` + `B62D57`, stratified across 2008–2025, ~30k families) containing all **2,907 humanoid families** of Step 0 — evaluated on both the full robotics universe and the humanoid sector alone (Section 1) |
| Which signals exist at publication ($t_0$), with zero leakage? | 4 static pillars: (1) classic ex-ante metadata & PCT, (2) originality, tech cycle time & causal CPC recombination (NR), (3) global examiner search reports (`TLS215` X/Y/I), (4) causal subfield momentum + zero-leakage Arts et al. lexical novelty, Kelly et al. semantic novelty & OpenAlex `PaECTER` science proximity (Section 3) |
| How much is gained after 2 years ($t_0 + 2\text{y}$)? | Multi-horizon checkpoint adding Age-Rescaled PageRank (Mariani et al. 2019) & CD disruption index (Funk & Owen-Smith 2017) strictly $\le t_0 + 2\text{y}$ (Section 3.5 & 4.4) |
| Does the model actually beat literature baselines out of time? | **Strict rolling out-of-time backtest** (`T = 2014–2017`) judged on **PR-AUC, Precision@Top10%, Recall@Top10%, and Lift** (Theory 6 p. 36) against random, classic metadata, rule-based scorecard, and single-signal baselines (Section 4) |
| Which signals drive the predictions (no black box)? | Standardized Logistic Regression signs + holdout Permutation Importance for the winning Boosting model (Section 5) |
| Which firms lead the 2022–2025 wave, and where do they sit on the Step 5 Quadrant? | Multi-aggregation firm scores (Mean quality, Expected breakthroughs, Top-3 concentration, Rule-based scorecard) across mature firms, 2023–2025 entrants, and a missing-player robotics extension (`AgiBot`, `Unitree`, `Boston Dynamics`), exported to `outputs/step4/future_score.csv` and `magic_quadrant_data.csv` (Sections 6–7) |

**Contents**

1. Setup, auto-export hooks & robotics training universe
2. Targets: breakthrough technological impact & international economic value
3. The 5-pillar feature stack (strictly causal at $t_0$, plus $t_0 + 2\text{y}$ dynamics)
4. Strict rolling out-of-time backtest & literature baseline comparisons
5. Model interpretability: what actually predicts breakthroughs?
6. Scoring the 2022–2025 humanoid firms & missing-player robotics extension
7. Step 5 Magic Quadrant bridge, sanity checks & deck answer


## 1. Setup, configuration & robotics training universe

Following our group decision (`docs/step4_plan_imma.md`) and **Theory 6**, the **training universe** is wider than the 2,907-family humanoid sector (which has only ~480–800 families in complete 5-year citation cohorts), while **firm scoring** stays strictly on the validated humanoid sector (plus a clearly flagged robotics extension for missing players like AgiBot, Unitree, and Boston Dynamics). All outputs (figures, tables, caches, HTML gallery) write into git-ignored `outputs/step4/`.


In [ ]:
import builtins as _builtins
import html as _html
import json
import re
import sys
import textwrap
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.inspection import permutation_importance
from sqlalchemy import and_, func, or_

sys.path.insert(0, str(Path("src").resolve()))
from future_score import (
    aggregate_firm_future_scores,
    bootstrap_metric_ci,
    compute_arts_lexical_novelty,
    compute_causal_recombination_novelty,
    compute_causal_subfield_momentum,
    compute_dynamic_network_at_age,
    compute_originality_and_tct,
    label_top_decile_by_cohort,
    make_classifiers,
    precision_recall_lift_at_k,
    rule_based_future_score,
    run_rolling_oot_backtest,
)

from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import (
    TLS201_APPLN,
    TLS206_PERSON,
    TLS207_PERS_APPLN,
    TLS209_APPLN_IPC,
    TLS211_PAT_PUBLN,
    TLS215_CITN_CATEG,
    TLS224_APPLN_CPC,
    TLS228_DOCDB_FAM_CITN,
)


In [ ]:
# ---------------------------------------------------------------- environment & parameters
ENV = "PROD"
RANDOM_STATE = 34
YEAR_MIN, YEAR_MAX = 2008, 2025          # 2008-2025 covers modern robotics + full burn-in before T=2014
LAST_COMPLETE_YEAR = 2022
HORIZON = 5                              # 5-year forward citation window (complete up to 2017)
LAST_LABEL_YEAR = LAST_COMPLETE_YEAR - HORIZON   # 2017
TEST_YEARS = [2014, 2015, 2016, 2017]    # strict rolling out-of-time holdout years
LOOKBACK = 5                             # train strictly on [T - LOOKBACK, T - 1]
APPLY_YEARS = (2022, 2025)               # recent wave scored for Step 4 Future Score (x-axis)
TOP_SHARE = 0.10                         # top-decile breakthrough target (Fleming 2001)
MIN_COHORT = 30
STRAT_PER_YEAR = 1_600                   # stratified non-sector robotics families per year (~28.8k + 2.9k sector)
CHUNK = 8_000
DUMMY_ID_FROM = 900_000_000

DATA_DIR = next((p for p in (Path("data") / ENV / "humanoid_robots_v2",
                             Path("..") / "tutorial_0" / "data" / ENV / "humanoid_robots_v2")
                 if (p / "dataset_metadata.json").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Run Step 0 on PROD first: data/PROD/humanoid_robots_v2/dataset_metadata.json not found")

WIDE_NET_FILE = Path("wide_net/outputs/wide_net/wide_net_families.csv")
SECTOR_IDS_FILE = Path(f"ids/sector_family_ids_v3_{ENV}.csv")
STEP3_DIR = Path("outputs/step3")
TEXT_NOVELTY_FILE = Path(f"outputs/t2_text/text_novelty_{ENV}_rule_v3.csv")
OPENALEX_FILE = STEP3_DIR / "science_proximity_openalex.csv"

OUT_DIR = Path("outputs/step4")
CACHE_DIR = OUT_DIR / "cache"
FIGURES_DIR = OUT_DIR / "figures"
TABLES_DIR = OUT_DIR / "tables"
for _d in (OUT_DIR, CACHE_DIR, FIGURES_DIR, TABLES_DIR):
    _d.mkdir(parents=True, exist_ok=True)
ANSWERS_FILE = OUT_DIR / "answers.txt"
ANSWERS_FILE.write_text("", encoding="utf-8")

# ---------------------------------------------------------------- auto-export hooks (git-ignored outputs/step4/)
_EXPORT_LOG = {"figures": [], "tables": [], "answers": []}


def _slug(text, default="item"):
    s = re.sub(r"[^a-zA-Z0-9]+", "_", str(text or "")).strip("_").lower()
    return s[:60] if s else default


def _fig_title(fig):
    if getattr(fig, "_suptitle", None) is not None and fig._suptitle.get_text().strip():
        return fig._suptitle.get_text().strip()
    for ax in fig.axes:
        t = ax.get_title().strip()
        if t:
            return t
    return "figure"


if not hasattr(plt, "_orig_show_step4"):
    plt._orig_show_step4 = plt.show


def _auto_save_show(*args, **kwargs):
    for fn in plt.get_fignums():
        fig = plt.figure(fn)
        if getattr(fig, "_step4_exported", False):
            continue
        fig._step4_exported = True
        idx = len(_EXPORT_LOG["figures"]) + 1
        title = _fig_title(fig)
        fname = f"{idx:02d}_{_slug(title, 'figure')}.png"
        fig.savefig(FIGURES_DIR / fname, dpi=150, bbox_inches="tight")
        _EXPORT_LOG["figures"].append({"file": f"figures/{fname}", "title": title})
    return plt._orig_show_step4(*args, **kwargs)


plt.show = _auto_save_show

if not hasattr(_builtins, "_orig_display_step4"):
    _base_disp = globals().get("display", getattr(_builtins, "display", None))
    if _base_disp is None:
        try:
            from IPython.display import display as _base_disp
        except Exception:
            _base_disp = print
    _builtins._orig_display_step4 = _base_disp


def display(*objs, **kwargs):
    for o in objs:
        if isinstance(o, (pd.DataFrame, pd.Series)):
            idx = len(_EXPORT_LOG["tables"]) + 1
            fname = f"{idx:02d}_table.csv"
            o.to_csv(TABLES_DIR / fname)
            _EXPORT_LOG["tables"].append({"file": f"tables/{fname}", "rows": int(len(o))})
    return _builtins._orig_display_step4(*objs, **kwargs)


def answer_box(question, headline, detail=""):
    _EXPORT_LOG["answers"].append({"question": question, "headline": headline, "detail": detail})
    with open(ANSWERS_FILE, "a", encoding="utf-8") as fh:
        fh.write(f"ANSWER · {question}\n{headline}\n{detail}\n" + "-" * 80 + "\n")
    print("-" * 80)
    print(f"ANSWER · {question}")
    for txt in (headline, detail):
        if txt:
            print(textwrap.fill(txt, width=80))
    print("-" * 80)


def load_step0(name):
    for suf, fn in ((".parquet", pd.read_parquet), (".pkl", pd.read_pickle)):
        if (DATA_DIR / f"{name}{suf}").exists():
            return fn(DATA_DIR / f"{name}{suf}")
    raise FileNotFoundError(f"{name} not in {DATA_DIR}")


def write_step4_report_index():
    (OUT_DIR / "run_manifest.json").write_text(json.dumps(_EXPORT_LOG, indent=2), encoding="utf-8")
    parts = [
        "<!doctype html><html><head><meta charset='utf-8'><title>Step 4 Future Score Outputs</title>",
        "<style>body{font-family:system-ui,sans-serif;max-width:1100px;margin:2rem auto;padding:0 1rem;color:#222}"
        "img{max-width:100%;border:1px solid #ddd;border-radius:6px;margin:0.5rem 0 1.5rem}"
        ".box{background:#f8f9fa;border-left:4px solid #e67e22;padding:0.8rem 1rem;margin:1rem 0}</style></head><body>",
        "<h1>Step 4 · Future Score &amp; Backtest Gallery</h1>",
    ]
    for a in _EXPORT_LOG["answers"]:
        parts.append(f"<div class='box'><strong>{_html.escape(a['question'])}</strong><br>{_html.escape(a['headline'])}<br><small>{_html.escape(a['detail'])}</small></div>")
    parts.append(f"<h2>Figures ({len(_EXPORT_LOG['figures'])})</h2>")
    for f in _EXPORT_LOG["figures"]:
        parts.append(f"<h3>{_html.escape(f['title'])}</h3><img src='{_html.escape(f['file'])}'>")
    parts.append(f"<h2>Tables ({len(_EXPORT_LOG['tables'])})</h2><ul>")
    for t in _EXPORT_LOG["tables"]:
        parts.append(f"<li><a href='{_html.escape(t['file'])}'>{_html.escape(t['file'])}</a> ({t['rows']} rows)</li>")
    parts.append("</ul></body></html>")
    idx_path = OUT_DIR / "index.html"
    idx_path.write_text("\n".join(parts), encoding="utf-8")
    print(f"Exported {len(_EXPORT_LOG['figures'])} figures, {len(_EXPORT_LOG['tables'])} tables -> {idx_path}")
    return idx_path


print(f"Step 4 configured | ENV={ENV} | Train/Test={YEAR_MIN}-{LAST_LABEL_YEAR} | Test years={TEST_YEARS} | Score={APPLY_YEARS}")


**Our answer for the deck (Steps 4 → 5 → 6).** Putting Step 3 (Present Score, living patent stock, and ownership transfers), Step 4 (Future Score), and Electrolux Jaffe technological proximity together yields a sharp strategic map for Electrolux:
- **Sole Mature Leader (Partner candidate): UBTECH Robotics** — `Present 87.2` (86/94 living families, residual stock `64.8`), `Future 71.9` (`n = 35`), and **#1 technological proximity to Electrolux (`0.431`)**. Main risk: heavy China concentration (`2.9%` PCT) and minimal European IP coverage.
- **Top Visionary (Strategic Acquisition / Equity / Western Partner candidate): Figure AI** — **#1 overall Future Score (`96.9`)**, `n = 44` recent families, `18.2%` PCT international protection, and **#1 technological proximity to Electrolux among all young entrants (`0.359`)** (sharing `B25J`, `G05D`, `H02J`, `G05B`, `G06V`). **Apptronik (`91.4`, `100%` PCT, `n = 5`)** is a high-quality smaller Western alternative.
- **Scale Chinese Challengers:** **Shanghai Fourier (`87.5`, `n = 91`, proximity `0.275`)** and **Guangzhou Ligong (`93.7`, `n = 92`, proximity `0.160`)** generate the highest volume of expected breakthroughs, but with near-zero PCT protection abroad (`0–3%`).
- **Fading Incumbents:** **Honda (`Present 86.7`)**, **Toyota**, **Kawada**, and **Tmsuk** have `0` recent humanoid filings and `0–12%` living patents; **Aldebaran (`Present 82.3`, `Future 54.3`, `n = 3`)** retains strong living European IP (`PAI 229.3`) but is flagged by `71%` ownership-transfer events and its 2025 insolvency / Maxvision asset sale.

**Limits that we declare.**
- **Training sample vs. sector:** Models are trained on a stratified 31,521-family robotics universe (`B25J` + `B62D57` + sector) so out-of-time intervals stay tight; the humanoid-only test slice (`n = 330`) has wider bootstrap intervals (`[0.246, 0.575]`), which we report openly.
- **Title-based sector boundary:** Firms patenting humanoids under generic titles (`AgiBot`, `Unitree`, `Boston Dynamics`) are under-counted in the strict sector and reported separately in Section 6.2.
- **2024–2025 publication lag:** Because of the 18-month publication lag, recent family counts and PCT filings for `2024–2025` are lower bounds, and `t0` probabilities predict *early technological breakthrough potential*, not audited commercial revenue.


### 1.1 Building the robotics training universe & cached PATSTAT queries

We combine **all 2,907 humanoid families** (Step 0, rule v3) with a **stratified robotics universe** (`wide_net_families.csv` if available, otherwise PATSTAT `B25J` manipulators + `B62D57` legged/walking robots, sampled up to 1,600 non-sector families per priority year across `2008–2025` with fixed seed 34). Every PATSTAT query is cached under `outputs/step4/cache/` so reruns take seconds.


In [ ]:
patstat = PatstatClient(env=ENV)
db = patstat.orm()
REAL = (TLS201_APPLN.ipr_type == "PI", TLS201_APPLN.appln_id < DUMMY_ID_FROM)


def in_chunks(ids, make_query, chunk=CHUNK):
    ids = [int(i) for i in pd.unique(pd.Series(list(ids)))]
    parts = [patstat.df(make_query(ids[s:s + chunk])) for s in range(0, len(ids), chunk)]
    parts = [p for p in parts if len(p)]
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()


def cached(name, fn, *args):
    p = CACHE_DIR / f"{name}.parquet"
    if p.exists():
        return pd.read_parquet(p)
    t0 = time.time()
    df = fn(*args)
    df.to_parquet(p, index=False)
    print(f"cached {name}: {len(df):,} rows in {time.time() - t0:.1f}s")
    return df


sector_families = load_step0("families")
sector_ids = set(sector_families["docdb_family_id"].astype(int))
if SECTOR_IDS_FILE.exists():
    sector_ids |= set(pd.read_csv(SECTOR_IDS_FILE)["docdb_family_id"].astype(int))


def build_universe():
    if WIDE_NET_FILE.exists():
        wn = pd.read_csv(WIDE_NET_FILE)
        wn = wn[wn["priority_year"].between(YEAR_MIN, YEAR_MAX)].copy()
        wn["in_sector_v3"] = wn["docdb_family_id"].isin(sector_ids)
    else:
        ipc_clean = func.replace(TLS209_APPLN_IPC.ipc_class_symbol, " ", "")
        q = (
            db.query(
                TLS201_APPLN.docdb_family_id.label("docdb_family_id"),
                func.min(TLS201_APPLN.earliest_filing_year).label("priority_year"),
            )
            .join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
            .filter(
                *REAL,
                or_(ipc_clean.like("B25J%"), ipc_clean.like("B62D57%")),
                TLS201_APPLN.earliest_filing_year.between(YEAR_MIN, YEAR_MAX),
            )
            .group_by(TLS201_APPLN.docdb_family_id)
        )
        wn = patstat.df(q)
        wn = wn[wn["priority_year"].between(YEAR_MIN, YEAR_MAX)].copy()
        wn["in_sector_v3"] = wn["docdb_family_id"].isin(sector_ids)

    # Keep ALL sector families + stratified sample of non-sector robotics families (<= STRAT_PER_YEAR per year)
    sec_part = sector_families[sector_families["priority_year"].between(YEAR_MIN, YEAR_MAX)][
        ["docdb_family_id", "priority_year"]
    ].drop_duplicates("docdb_family_id").assign(in_sector_v3=True)

    non_sec = wn[~wn["docdb_family_id"].isin(sector_ids)][["docdb_family_id", "priority_year"]].drop_duplicates("docdb_family_id")
    sampled_parts = []
    for yr, grp in non_sec.groupby("priority_year"):
        sampled_parts.append(grp.sample(min(len(grp), STRAT_PER_YEAR), random_state=RANDOM_STATE))
    non_sec_sample = pd.concat(sampled_parts, ignore_index=True).assign(in_sector_v3=False) if sampled_parts else pd.DataFrame(columns=["docdb_family_id", "priority_year", "in_sector_v3"])

    univ = pd.concat([sec_part, non_sec_sample], ignore_index=True).drop_duplicates("docdb_family_id")
    univ["docdb_family_id"] = univ["docdb_family_id"].astype(int)
    univ["priority_year"] = univ["priority_year"].astype(int)
    return univ


net = cached("robotics_universe", build_universe).set_index("docdb_family_id")
FAMILIES = net.index.tolist()
print(f"Training universe: {len(net):,} families ({int(net['in_sector_v3'].sum()):,} humanoid sector | {int((~net['in_sector_v3']).sum()):,} robotics controls)")
print("Families by priority year (2008-2025):")
display(net.groupby("priority_year").agg(total=("in_sector_v3", "size"), humanoid=("in_sector_v3", "sum")).T)


**Our reading of the training universe.** The stratified universe combines **2,721 humanoid families** (`2008–2025`, all rule-v3 families in the window) with **28,800 robotics control families** (`B25J` + `B62D57`, 1,600 per year), for **31,521 families** in total. This solves the small-sample bottleneck of training on humanoids alone (~480–800 families in complete cohorts) while keeping the evaluation anchored on both the full robotics universe and the humanoid sector alone.


## 2. Basic family attributes & dual future targets

We query PATSTAT for applications, CPC/IPC subclasses, and 5-year forward citations across the universe, then build two future targets on the complete `2008–2017` cohorts:
1. **Primary target (`valuable`):** Top 10% of 5-year forward citations within the family's `(priority_year, main_subclass)` cohort (Fleming 2001 tail-breakthrough target).
2. **Secondary economic target (`intl_valuable`):** Family expands to `>= 2` patent offices OR obtains a US/EP grant — testing whether our algorithm predicts genuine international economic value rather than domestic citation circles.


In [ ]:
def fetch_applications(ids):
    return in_chunks(ids, lambda part: db.query(
        TLS201_APPLN.docdb_family_id,
        TLS201_APPLN.appln_id,
        TLS201_APPLN.appln_auth,
        TLS201_APPLN.earliest_filing_year,
        TLS201_APPLN.earliest_filing_date,
        TLS201_APPLN.appln_filing_year,
        TLS201_APPLN.appln_filing_date,
        TLS201_APPLN.granted,
        TLS201_APPLN.nb_inventors,
        TLS201_APPLN.nb_applicants,
    ).filter(*REAL, TLS201_APPLN.docdb_family_id.in_(part)))


def fetch_subclasses(ids):
    cpc = in_chunks(ids, lambda part: db.query(
        TLS201_APPLN.docdb_family_id,
        TLS224_APPLN_CPC.cpc_class_symbol.label("symbol"),
    ).join(TLS224_APPLN_CPC, TLS201_APPLN.appln_id == TLS224_APPLN_CPC.appln_id)
     .filter(*REAL, TLS201_APPLN.docdb_family_id.in_(part)).distinct())
    cpc["code"] = cpc["symbol"].astype(str).str.replace(" ", "", regex=False).str[:4]
    cpc = cpc[~cpc["code"].str.startswith("Y") & (cpc["code"].str.len() == 4)][["docdb_family_id", "code"]].drop_duplicates()

    missing_ids = list(set(ids) - set(cpc["docdb_family_id"].unique()))
    if missing_ids:
        ipc = in_chunks(missing_ids, lambda part: db.query(
            TLS201_APPLN.docdb_family_id,
            TLS209_APPLN_IPC.ipc_class_symbol.label("symbol"),
        ).join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
         .filter(*REAL, TLS201_APPLN.docdb_family_id.in_(part)).distinct())
        ipc["code"] = ipc["symbol"].astype(str).str.replace(" ", "", regex=False).str[:4]
        ipc = ipc[ipc["code"].str.len() == 4][["docdb_family_id", "code"]].drop_duplicates()
        return pd.concat([cpc, ipc], ignore_index=True).drop_duplicates()
    return cpc


def fetch_forward_cites(ids):
    return in_chunks(ids, lambda part: db.query(
        TLS228_DOCDB_FAM_CITN.cited_docdb_family_id.label("docdb_family_id"),
        TLS228_DOCDB_FAM_CITN.docdb_family_id.label("citing_family"),
        func.min(TLS201_APPLN.earliest_filing_year).label("citing_year"),
    ).join(TLS201_APPLN, TLS201_APPLN.docdb_family_id == TLS228_DOCDB_FAM_CITN.docdb_family_id)
     .filter(*REAL,
             TLS228_DOCDB_FAM_CITN.cited_docdb_family_id.in_(part),
             TLS228_DOCDB_FAM_CITN.docdb_family_id != TLS228_DOCDB_FAM_CITN.cited_docdb_family_id)
     .group_by(TLS228_DOCDB_FAM_CITN.cited_docdb_family_id, TLS228_DOCDB_FAM_CITN.docdb_family_id))


apps = cached("applications", fetch_applications, FAMILIES)
fam_codes = cached("family_subclasses", fetch_subclasses, FAMILIES)
fwd = cached("forward_citations", fetch_forward_cites, FAMILIES)

# Main 4-char subclass per family
main_sub = (fam_codes.groupby(["docdb_family_id", "code"]).size().rename("n").reset_index()
            .sort_values(["docdb_family_id", "n", "code"], ascending=[True, False, True])
            .drop_duplicates("docdb_family_id").set_index("docdb_family_id")["code"])
net["main_subclass"] = net.index.map(main_sub).fillna("none")

# 5-year forward citations target
fwd_m = fwd.merge(net[["priority_year"]], left_on="docdb_family_id", right_index=True)
in_5y = fwd_m["citing_year"] <= fwd_m["priority_year"] + HORIZON
net["fwd_cites_5y"] = net.index.map(fwd_m[in_5y].groupby("docdb_family_id")["citing_family"].nunique()).fillna(0).astype(int)

labelled_mask = net["priority_year"].between(YEAR_MIN, LAST_LABEL_YEAR)
net["valuable"] = np.nan
net.loc[labelled_mask, "valuable"], net.loc[labelled_mask, "cohort"] = label_top_decile_by_cohort(
    net.loc[labelled_mask], value_col="fwd_cites_5y", top_share=TOP_SHARE, min_cohort=MIN_COHORT
)

# Secondary economic target: reaches >= 2 distinct national offices OR US/EP grant
nat_apps = apps[apps["appln_auth"].astype(str).str.strip() != "WO"].copy()
nat_apps["appln_auth"] = nat_apps["appln_auth"].astype(str).str.strip()
nat_apps["is_granted"] = nat_apps["granted"].astype(str).str.strip().str.upper().isin(["Y", "1", "TRUE"])
total_offices = nat_apps.groupby("docdb_family_id")["appln_auth"].nunique()
us_ep_granted = nat_apps[nat_apps["is_granted"] & nat_apps["appln_auth"].isin(["US", "EP"])].groupby("docdb_family_id").size() > 0
net["total_offices"] = net.index.map(total_offices).fillna(1).astype(int)
net["intl_valuable"] = np.nan
net.loc[labelled_mask, "intl_valuable"] = (
    (net.loc[labelled_mask, "total_offices"] >= 2) | net.loc[labelled_mask].index.isin(us_ep_granted[us_ep_granted].index)
).astype(int)

target_summary = net.loc[labelled_mask].groupby("priority_year").agg(
    families=("valuable", "size"),
    humanoid=("in_sector_v3", "sum"),
    top10_rate_pct=("valuable", lambda s: s.mean() * 100),
    intl_rate_pct=("intl_valuable", lambda s: s.mean() * 100),
    median_cites_5y=("fwd_cites_5y", "median"),
    p90_cites_5y=("fwd_cites_5y", lambda s: s.quantile(0.90)),
)
print(f"Labelled families ({YEAR_MIN}-{LAST_LABEL_YEAR}): {int(labelled_mask.sum()):,} "
      f"(valuable top-10%: {int(net['valuable'].sum()):,} = {net['valuable'].mean():.1%} | "
      f"international economic target: {int(net['intl_valuable'].sum()):,} = {net['intl_valuable'].mean():.1%})")
display(target_summary.round(2))


**Our reading of the targets.** Across the complete `2008–2017` cohorts (**16,569 labelled families**), **1,768 families (10.7%)** reach the primary **top-decile 5-year breakthrough target** within their `(priority_year, main_subclass)` cohort, while **4,802 (29.0%)** meet the secondary international economic target (`>= 2` offices or a US/EP grant). The ~10.7% base rate confirms why raw accuracy is misleading (`Theory 6`, p. 36) and why we judge models on **PR-AUC, Precision@Top10%, and Lift**.


## 3. The 5-pillar feature stack (strictly ex-ante at $t_0$ + $t_0 + 2\text{y}$ dynamics)

Every static feature uses **only information available at filing/publication ($t_0$)** — never grants, never total later family size, never forward citations. We register each feature with its theoretical source, set (`classic` benchmark vs. `ours` proposed), and expected direction for the zero-training Rule-Based Scorecard (`Theory 6`, p. 25).


In [ ]:
SIGNALS = {}
signals = pd.DataFrame(index=net.index)


def register(name, series, group, timing, direction, source):
    signals[name] = pd.Series(series).reindex(net.index).astype(float)
    SIGNALS[name] = {"set": group, "timing": timing, "direction": direction, "source": source}


# --- Pillar 1: Classic ex-ante metadata & early economic commitment (at t0) ---
apps_dt = apps.copy()
apps_dt["appln_auth"] = apps_dt["appln_auth"].astype(str).str.strip()
apps_dt["earliest_dt"] = pd.to_datetime(apps_dt["earliest_filing_date"], errors="coerce")
apps_dt["filing_dt"] = pd.to_datetime(apps_dt["appln_filing_date"], errors="coerce")
apps_dt["earliest_dt"] = apps_dt["earliest_dt"].where(apps_dt["earliest_dt"].dt.year < 9000)
apps_dt["filing_dt"] = apps_dt["filing_dt"].where(apps_dt["filing_dt"].dt.year < 9000)
fam_first_dt = apps_dt.groupby("docdb_family_id")["earliest_dt"].min()
apps_dt["days_from_prio"] = (apps_dt["filing_dt"] - apps_dt["docdb_family_id"].map(fam_first_dt)).dt.days

# Offices reached within 12 months (strictly early family size, Harhoff et al. 2003)
early_nat = apps_dt[(apps_dt["appln_auth"] != "WO") & (apps_dt["days_from_prio"].fillna(0) <= 366)]
register("early_offices_12m", early_nat.groupby("docdb_family_id")["appln_auth"].nunique().fillna(1),
         "classic", "static", +1, "Harhoff et al. (2003)")
register("has_pct", apps_dt[apps_dt["appln_auth"] == "WO"].groupby("docdb_family_id").size().gt(0).reindex(net.index).fillna(False).astype(float),
         "classic", "static", +1, "Theory 6 p. 12 (PCT route)")
register("n_inventors", apps_dt.groupby("docdb_family_id")["nb_inventors"].max().fillna(0),
         "classic", "static", +1, "Theory 5 p. 23; Lee et al. (2018)")
register("n_applicants", apps_dt.groupby("docdb_family_id")["nb_applicants"].max().fillna(1),
         "classic", "static", +1, "Theory 5 p. 43 (co-applicants)")
register("scope_cpc4", fam_codes.groupby("docdb_family_id")["code"].nunique().fillna(1),
         "classic", "static", +1, "Lerner (1994); Lai et al. (2026)")

# University co-applicant flag (TLS207 + TLS206)
def fetch_univ_flag(ids):
    df = in_chunks(ids, lambda part: db.query(
        TLS201_APPLN.docdb_family_id,
        TLS206_PERSON.psn_sector,
    ).join(TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id)
     .join(TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id)
     .filter(*REAL, TLS207_PERS_APPLN.applt_seq_nr > 0, TLS201_APPLN.docdb_family_id.in_(part)).distinct())
    df["is_univ"] = df["psn_sector"].astype(str).str.upper().str.contains("UNIVERSITY")
    return df.groupby("docdb_family_id")["is_univ"].any().reset_index()


univ_flag = cached("univ_coapp", fetch_univ_flag, FAMILIES).set_index("docdb_family_id")["is_univ"].astype(float)
register("has_university", univ_flag.reindex(net.index).fillna(0.0), "classic", "static", 0, "Step 1 & 2 (academic vs corporate)")

# First filing office controls (Lemley & Sampat 2012)
first_off = (apps_dt[apps_dt["appln_auth"] != "WO"].sort_values(["docdb_family_id", "filing_dt"])
             .drop_duplicates("docdb_family_id").set_index("docdb_family_id")["appln_auth"])
net["first_office"] = net.index.map(first_off).fillna("OTHER")
for _off in ("CN", "US", "JP", "KR", "EP"):
    register(f"office_{_off}", (net["first_office"] == _off).astype(float), "classic", "static", 0, "Lemley & Sampat (2012)")

print(f"Pillar 1 (classic ex-ante) registered: {len(SIGNALS)} features")


In [ ]:
# --- Pillar 2: Backward citations, Originality, Tech Cycle Time & Causal Recombination (NR) ---
def fetch_backward_cites(ids):
    b = in_chunks(ids, lambda part: db.query(
        TLS228_DOCDB_FAM_CITN.docdb_family_id.label("docdb_family_id"),
        TLS228_DOCDB_FAM_CITN.cited_docdb_family_id.label("cited_family"),
        func.min(TLS201_APPLN.earliest_filing_year).label("cited_year"),
    ).join(TLS201_APPLN, TLS201_APPLN.docdb_family_id == TLS228_DOCDB_FAM_CITN.cited_docdb_family_id)
     .filter(*REAL,
             TLS228_DOCDB_FAM_CITN.docdb_family_id.in_(part),
             TLS228_DOCDB_FAM_CITN.docdb_family_id != TLS228_DOCDB_FAM_CITN.cited_docdb_family_id)
     .group_by(TLS228_DOCDB_FAM_CITN.docdb_family_id, TLS228_DOCDB_FAM_CITN.cited_docdb_family_id))
    if not len(b):
        return pd.DataFrame(columns=["docdb_family_id", "cited_family", "cited_year", "cited_subclass"])
    cited_ids = b["cited_family"].unique().tolist()
    c_ipc = in_chunks(cited_ids, lambda part: db.query(
        TLS201_APPLN.docdb_family_id.label("cited_family"),
        TLS209_APPLN_IPC.ipc_class_symbol.label("symbol"),
    ).join(TLS209_APPLN_IPC, TLS201_APPLN.appln_id == TLS209_APPLN_IPC.appln_id)
     .filter(*REAL, TLS201_APPLN.docdb_family_id.in_(part)).distinct())
    c_ipc["cited_subclass"] = c_ipc["symbol"].astype(str).str.replace(" ", "", regex=False).str[:4]
    c_ipc = (c_ipc[c_ipc["cited_subclass"].str.len() == 4]
             .groupby(["cited_family", "cited_subclass"]).size().rename("n").reset_index()
             .sort_values(["cited_family", "n", "cited_subclass"], ascending=[True, False, True])
             .drop_duplicates("cited_family")[["cited_family", "cited_subclass"]])
    return b.merge(c_ipc, on="cited_family", how="left")


bwd = cached("backward_citations", fetch_backward_cites, FAMILIES)
orig_tct = compute_originality_and_tct(bwd, net["priority_year"])
register("n_backward_cites", orig_tct["n_bwd_cited_fams"].reindex(net.index).fillna(0.0),
         "classic", "static", +1, "Harhoff et al. (2003)")
register("originality", orig_tct["originality"],
         "ours", "static", +1, "Trajtenberg et al. (1997); Lai et al. (2026)")
register("tech_cycle_time", orig_tct["tech_cycle_time"],
         "ours", "static", -1, "Lai et al. (2026) (shorter = faster frontier)")

# Causal recombination novelty (NR, Fleming 2001; Verhoeven et al. 2016)
nr_df = compute_causal_recombination_novelty(fam_codes, net["priority_year"])
register("novel_pair_share_nr", nr_df["novel_pair_share_nr"].reindex(net.index).fillna(0.0),
         "ours", "static", +1, "Fleming (2001); Verhoeven et al. (2016)")
register("has_novel_pair_nr", nr_df["has_novel_pair_nr"].reindex(net.index).fillna(0.0),
         "ours", "static", +1, "Fleming (2001); Theory 6 p. 18")

# Causal hot-subfield momentum (strictly before filing year t)
mom = compute_causal_subfield_momentum(net, window=3)
register("subfield_momentum", mom, "ours", "static", +1, "Theory 6 p. 23 (hot subfields)")

print(f"Pillar 2 + subfield momentum registered | novel pair share > 0: {(signals['has_novel_pair_nr'] > 0).mean():.1%}")


In [ ]:
# --- Pillar 3: Global examiner search reports (TLS215 X/Y/I across all offices) ---
def fetch_search_reports(ids):
    app_map = apps[["appln_id", "docdb_family_id"]].drop_duplicates()
    aids = app_map["appln_id"].astype(int).tolist()
    pubs = in_chunks(aids, lambda part: db.query(
        TLS211_PAT_PUBLN.pat_publn_id, TLS211_PAT_PUBLN.appln_id, TLS211_PAT_PUBLN.publn_auth
    ).filter(TLS211_PAT_PUBLN.appln_id.in_(part)))
    if not len(pubs):
        return pd.DataFrame(columns=["docdb_family_id", "has_report", "has_xy", "xy_share", "n_cats"])
    pubs = pubs.merge(app_map, on="appln_id")
    cats = in_chunks(pubs["pat_publn_id"].astype(int).tolist(), lambda part: db.query(
        TLS215_CITN_CATEG.pat_publn_id, TLS215_CITN_CATEG.citn_categ
    ).filter(TLS215_CITN_CATEG.pat_publn_id.in_(part)))
    if not len(cats):
        return pd.DataFrame(columns=["docdb_family_id", "has_report", "has_xy", "xy_share", "n_cats"])
    cats = cats.merge(pubs[["pat_publn_id", "docdb_family_id"]], on="pat_publn_id")
    cats["is_xy"] = cats["citn_categ"].astype(str).str.contains("[XYI]", regex=True)
    return (cats.groupby("docdb_family_id")
            .agg(has_report=("is_xy", lambda s: 1.0),
                 has_xy=("is_xy", "max"),
                 xy_share=("is_xy", "mean"),
                 n_cats=("is_xy", "size"))
            .reset_index())


sr = cached("search_reports_tls215", fetch_search_reports, FAMILIES).set_index("docdb_family_id")
register("has_search_report", sr["has_report"].reindex(net.index).fillna(0.0),
         "ours", "static", 0, "TLS215 coverage flag")
register("examiner_has_xy", sr["has_xy"].astype(float),
         "ours", "static", -1, "Czarnitzki et al. (2011); Theory 6 p. 12 (missing if no report)")
register("examiner_xy_share", sr["xy_share"],
         "ours", "static", -1, "Czarnitzki et al. (2011); Lemley & Sampat (2012)")

# Validation check on 2015-2017 labelled families: do families WITHOUT X/Y get cited more?
_sr_check = net[net["priority_year"].between(2015, 2017)].join(signals[["has_search_report", "examiner_has_xy"]])
_sr_with = _sr_check[_sr_check["has_search_report"] == 1]
if len(_sr_with):
    print("Examiner X/Y validation on 2015-2017 families with search reports:")
    display(_sr_with.groupby("examiner_has_xy").agg(
        families=("fwd_cites_5y", "size"),
        mean_cites_5y=("fwd_cites_5y", "mean"),
        top10_rate_pct=("valuable", lambda s: s.mean() * 100),
        intl_rate_pct=("intl_valuable", lambda s: s.mean() * 100),
    ).round(2))


**Our reading of the examiner's opinion (`TLS215`).** On `2015–2017` families with search reports, families with an `X`/`Y`/`I` citation (`n = 3,134`) initially show slightly higher *unadjusted* mean 5-year citations (`8.00` vs. `6.43`, top-10% share `13.2%` vs. `10.3%`, international rate `30.7%` vs. `15.9%`) because crowded, commercially active filings attract both heavier filing abroad and tighter examiner scrutiny. Once we control for office, backward citations, and scope in the multivariate model (Section 5), `examiner_xy_share` carries the expected **negative sign (`-0.054`)**: heavier blocking prior art against an application signals narrower novelty.


In [ ]:
# --- Pillar 4: Sector-only Text Novelty (Arts zero-leakage + Kelly SBERT), Readability & OpenAlex PaECTER Science ---
SECTOR_ONLY = []

# 4a. Kelly et al. (2021) semantic novelty from Notebook 21
if TEXT_NOVELTY_FILE.exists():
    _t21 = pd.read_csv(TEXT_NOVELTY_FILE).drop_duplicates("docdb_family_id").set_index("docdb_family_id")
    _col = "novelty_rank_in_year" if "novelty_rank_in_year" in _t21.columns else _t21.columns[0]
    register("kelly_semantic_novelty_rank", _t21[_col], "ours", "static", +1, "Kelly et al. (2021); Notebook 21")
    SECTOR_ONLY.append("kelly_semantic_novelty_rank")

# 4b. Arts, Hou & Gomez (2021) ZERO-LEAKAGE lexical novelty + Lai et al. (2026) readability on sector abstracts
_abs_df = pd.DataFrame()
for _cand in (Path("outputs/t2_text/sector_abstracts_PROD_rule_v3.csv"), Path("outputs/step0/prod/02_net_abstracts.csv.gz")):
    if _cand.exists():
        _tmp = pd.read_csv(_cand)
        _acol = next((c for c in ("abstract", "appln_abstract", "abstract_text") if c in _tmp.columns), None)
        if _acol and "docdb_family_id" in _tmp.columns:
            _abs_df = (_tmp[_tmp["docdb_family_id"].isin(sector_ids)][["docdb_family_id", _acol]]
                       .dropna().drop_duplicates("docdb_family_id").rename(columns={_acol: "text"}))
            break

if len(_abs_df):
    _abs_df = _abs_df.merge(sector_families[["docdb_family_id", "priority_year"]], on="docdb_family_id")
    arts_df = compute_arts_lexical_novelty(_abs_df, text_col="text", year_col="priority_year", lookback_years=5)
    arts_df.to_csv(OUT_DIR / "arts_lexical_novelty.csv")
    register("arts_novelty_rank", arts_df["arts_novelty_rank_in_year"], "ours", "static", +1, "Arts, Hou & Gomez (2021) — zero leakage")
    register("abstract_long_word_share", arts_df["abstract_long_word_share"], "ours", "static", +1, "Lai et al. (2026) readability")
    SECTOR_ONLY += ["arts_novelty_rank", "abstract_long_word_share"]

# 4c. OpenAlex + PaECTER causal implicit science linkage from Step 3 Section 3.6
if OPENALEX_FILE.exists():
    _oa = pd.read_csv(OPENALEX_FILE).drop_duplicates("docdb_family_id").set_index("docdb_family_id")
    register("openalex_science_rank", _oa["science_rank_in_year"], "ours", "static", +1, "Theory 6 pp. 21-22 (OpenAlex + PaECTER)")
    register("openalex_fundamental_match", _oa["fundamental_match"].astype(float), "ours", "static", +1, "Theory 6 p. 22 (fast-rising papers)")
    SECTOR_ONLY += ["openalex_science_rank", "openalex_fundamental_match"]

# --- Pillar 5: Dynamic multi-horizon network signals strictly at t0 + 2 years (Mariani et al. 2019; Funk & Owen-Smith 2017) ---
dyn_2y = cached("dynamic_network_2y", lambda _: compute_dynamic_network_at_age(fwd, bwd, net["priority_year"], age_limit=2), None)
if "docdb_family_id" in dyn_2y.columns:
    dyn_2y = dyn_2y.set_index("docdb_family_id")
else:
    dyn_2y.index = net.index[:len(dyn_2y)]
DYNAMIC_2Y = ["early_cites_2y", "pagerank_age_ratio_2y", "cd_index_2y"]
register("early_cites_2y", dyn_2y["early_cites_2y"], "dynamic_2y", "dynamic", +1, "Lai et al. (2026) t0+2y")
register("pagerank_age_ratio_2y", dyn_2y["pagerank_age_ratio_2y"], "dynamic_2y", "dynamic", +1, "Mariani, Medo & Lafond (2019)")
register("cd_index_2y", dyn_2y["cd_index_2y"], "dynamic_2y", "dynamic", +1, "Funk & Owen-Smith (2017)")

sig_summary = pd.DataFrame(SIGNALS).T
sig_summary["coverage (%)"] = (signals.notna().mean() * 100).round(1)
print(f"Total registered signals: {len(SIGNALS)} (sector-only text/science: {len(SECTOR_ONLY)} | dynamic t0+2y: {len(DYNAMIC_2Y)})")
display(sig_summary)


## 4. Strict rolling out-of-time backtest & literature baselines

Following **Theory 6 (pp. 35–37)** and improving on Lai, Wei & Hwang (2026) (who used random CV on artificially balanced extremes), we run a **strict rolling out-of-time backtest** across `T = 2014, 2015, 2016, 2017` (training strictly on `[T - 5, T - 1]`, testing on `T`, on the real unbalanced distribution). We judge every model on **PR-AUC (Average Precision), ROC-AUC, Precision@Top10%, Recall@Top10%, and Lift@Top10%** with 95% bootstrap confidence intervals, comparing:
1. **Random base rate**
2. **Rule-Based Future Scorecard** (`Theory 6` p. 25, zero training)
3. **Single-signal literature baselines** (`early_offices_12m`, `originality`, `novel_pair_share_nr`, `pagerank_age_ratio_2y`)
4. **Benchmark ($t_0$):** classic metadata only (`Harhoff et al. / Lee et al.`)
5. **Proposed ($t_0$):** classic + originality + TCT + causal recombination + `TLS215` examiner opinion + subfield momentum
6. **Proposed ($t_0 + 2\text{y}$):** adds Age-Rescaled PageRank + CD index + early 2-year citations
7. **Robustness:** secondary international economic target (`intl_valuable`) + **Humanoid-sector-only ablation** quantifying the lift of Arts zero-leakage novelty, Kelly SBERT novelty, and OpenAlex `PaECTER` science proximity!


In [ ]:
CLASSIC = [s for s, d in SIGNALS.items() if d["set"] == "classic"]
OURS_UNIV = [s for s, d in SIGNALS.items() if d["set"] == "ours" and s not in SECTOR_ONLY]
FEATURE_SETS = {
    "1_benchmark_t0 (classic)": CLASSIC,
    "2_proposed_t0 (classic + ours)": CLASSIC + OURS_UNIV,
    "3_proposed_t0+2y (+ PageRank/CD)": CLASSIC + OURS_UNIV + DYNAMIC_2Y,
}

# 1. Rolling out-of-time backtest on PRIMARY target (top-10% 5-year citations)
preds = run_rolling_oot_backtest(
    signals, net["valuable"], net["priority_year"], net["in_sector_v3"],
    feature_sets=FEATURE_SETS, test_years=TEST_YEARS, lookback=LOOKBACK, random_state=RANDOM_STATE,
)
preds.to_csv(OUT_DIR / "backtest_predictions.csv", index=False)

# Zero-training Rule-Based Scorecard on the same test families
_rule_dirs = {k: d["direction"] for k, d in SIGNALS.items() if d["direction"] != 0 and k not in SECTOR_ONLY and k not in DYNAMIC_2Y}
signals["rule_based_t0"] = rule_based_future_score(signals, _rule_dirs, year_series=net["priority_year"])

test_fids = preds["docdb_family_id"].drop_duplicates().to_numpy()
y_test_all = net.loc[test_fids, "valuable"].astype(int).to_numpy()
sec_test_mask = net.loc[test_fids, "in_sector_v3"].to_numpy(dtype=bool)


def eval_table(pred_df, subset_label="All robotics", only_sector=False):
    rows = []
    sub_all = pred_df[pred_df["in_sector"]] if only_sector else pred_df
    for (fset, mod), g in sub_all.groupby(["feature_set", "model"]):
        m = precision_recall_lift_at_k(g["y"], g["score"], share=TOP_SHARE)
        pr_pt, pr_lo, pr_hi = bootstrap_metric_ci(g["y"], g["score"], "PR_AUC", n_boot=300, seed=RANDOM_STATE)
        rows.append({
            "scope": subset_label, "feature_set": fset, "model": mod,
            "n_test": len(g), "base_rate (%)": m["base_rate"] * 100,
            "PR_AUC": pr_pt, "PR_AUC_95CI": f"[{pr_lo:.3f}, {pr_hi:.3f}]",
            "ROC_AUC": m["ROC_AUC"], "P@10% (%)": m["P_at_10pct"] * 100,
            "R@10% (%)": m["R_at_10pct"] * 100, "Lift@10%": m["Lift_at_10pct"],
        })
    return pd.DataFrame(rows).sort_values(["feature_set", "PR_AUC"], ascending=[True, False])


bt_all = eval_table(preds, "Robotics universe", only_sector=False)
bt_sec = eval_table(preds, "Humanoid sector only", only_sector=True)

# Add Rule-Based & Single-Signal Baselines on the exact same test families
base_rows = []
for bname, bcol in [
    ("Rule-Based Scorecard (t0, zero training)", "rule_based_t0"),
    ("Single signal: early_offices_12m", "early_offices_12m"),
    ("Single signal: originality", "originality"),
    ("Single signal: novel_pair_share_nr", "novel_pair_share_nr"),
    ("Single signal: PageRank age-rescaled (t0+2y)", "pagerank_age_ratio_2y"),
]:
    if bcol not in signals.columns:
        continue
    s_vec = signals.loc[test_fids, bcol].to_numpy()
    m = precision_recall_lift_at_k(y_test_all, s_vec, share=TOP_SHARE)
    base_rows.append({
        "scope": "Robotics universe", "feature_set": "0_baselines", "model": bname,
        "n_test": int(np.isfinite(s_vec).sum()), "base_rate (%)": m["base_rate"] * 100,
        "PR_AUC": m["PR_AUC"], "PR_AUC_95CI": "—", "ROC_AUC": m["ROC_AUC"],
        "P@10% (%)": m["P_at_10pct"] * 100, "R@10% (%)": m["R_at_10pct"] * 100, "Lift@10%": m["Lift_at_10pct"],
    })
baselines_df = pd.DataFrame(base_rows).sort_values("PR_AUC", ascending=False)

backtest_summary = pd.concat([baselines_df, bt_all, bt_sec], ignore_index=True)
backtest_summary.to_csv(OUT_DIR / "backtest_summary.csv", index=False)
print("=== Rolling Out-of-Time Backtest Results (T = 2014-2017) ===")
display(backtest_summary.round(3))

# Plot PR-AUC & Lift@10% comparison
def _plot_backtest():
    best_by_set = bt_all.sort_values("PR_AUC", ascending=False).drop_duplicates("feature_set")
    comp = pd.concat([baselines_df[["model", "PR_AUC", "Lift@10%"]].rename(columns={"model": "label"}),
                      best_by_set.assign(label=best_by_set["feature_set"] + " · " + best_by_set["model"])[["label", "PR_AUC", "Lift@10%"]]],
                     ignore_index=True).sort_values("PR_AUC", ascending=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.2, 4.6))
    cols = ["#e67e22" if "proposed" in l else ("#2c7bb6" if "benchmark" in l else "#95a5a6") for l in comp["label"]]
    ax1.barh(comp["label"], comp["PR_AUC"], color=cols)
    ax1.axvline(y_test_all.mean(), color="#c0392b", ls="--", lw=1.2, label=f"Random ({y_test_all.mean():.3f})")
    for i, v in enumerate(comp["PR_AUC"]):
        ax1.text(v + 0.006, i, f"{v:.3f}", va="center", fontsize=8.5)
    ax1.set_xlabel("PR-AUC (Average Precision, out-of-time 2014–2017)")
    ax1.set_title("PR-AUC vs. Literature Baselines")
    ax1.legend(loc="lower right", fontsize=8)
    ax1.spines[["top", "right"]].set_visible(False)

    ax2.barh(comp["label"], comp["Lift@10%"], color=cols)
    ax2.axvline(1.0, color="#c0392b", ls="--", lw=1.2, label="Random (1.0×)")
    for i, v in enumerate(comp["Lift@10%"]):
        ax2.text(v + 0.06, i, f"{v:.2f}×", va="center", fontsize=8.5)
    ax2.set_yticks([])
    ax2.set_xlabel("Lift in Top-10% Shortlist (× over base rate)")
    ax2.set_title("Precision Lift in Top 10%")
    ax2.legend(loc="lower right", fontsize=8)
    ax2.spines[["top", "right"]].set_visible(False)
    plt.suptitle("Section 4 · Strict Out-of-Time Backtest (T = 2014–2017)", y=1.02, fontweight="bold")
    plt.tight_layout()
    plt.show()

_plot_backtest()
del _plot_backtest


**Our reading of the strict rolling out-of-time backtest (`T = 2014–2017`, base rate 10.7%).**
- **Every proposed model cleanly beats all static baselines out of time:** On the full robotics holdout (`n = 6,730`), the zero-training **Rule-Based Scorecard** (`PR-AUC 0.236`, `Lift 2.54×`) already outperforms every single static signal (`originality 0.201`, `early_offices_12m 0.184`, `novel_pair_share_nr 0.112`). Adding our proposed ex-ante signals (`2_proposed_t0`, Random Forest) lifts **PR-AUC from `0.314` (classic benchmark) to `0.335`** `[0.304, 0.369]` (**ROC-AUC `0.787`**, **Lift@10% `3.51×`** — finding 3.5× more breakthroughs in the top decile than random).
- **Stronger transfer on humanoid robots alone (`n = 330`):** On humanoid test families, the proposed `t0` Random Forest jumps from **`0.327` (classic benchmark) to `0.394` PR-AUC** (`+0.067`, ROC-AUC `0.770`, Lift `3.23×`).
- **Multi-horizon `t0 + 2y` jump (improving Mariani et al. 2019 & Lai et al. 2026):** Adding dynamic 2-year citation signals (Age-Rescaled PageRank + CD disruption index + early citations) pushes Logistic Regression to **PR-AUC `0.745`** `[0.713, 0.779]` (**ROC `0.953`**, **Lift `6.64×`**; **`0.742` / `6.77×`** on humanoids only) — decisively beating **Age-Rescaled PageRank alone (`0.648`, `5.89×`)** by nearly `+0.10` PR-AUC.


In [ ]:
# --- Secondary Economic Target Check (intl_valuable) & Humanoid Sector Text+OpenAlex Ablation ---
preds_intl = run_rolling_oot_backtest(
    signals, net["intl_valuable"], net["priority_year"], net["in_sector_v3"],
    feature_sets={"1_benchmark_t0": CLASSIC, "2_proposed_t0": CLASSIC + OURS_UNIV},
    test_years=TEST_YEARS, lookback=LOOKBACK, random_state=RANDOM_STATE,
)
intl_summary = eval_table(preds_intl, "International target (all)", only_sector=False)
print("=== Robustness Check: Secondary Economic Target (Multi-Office or US/EP Grant) ===")
display(intl_summary.round(3))

# Humanoid-sector-only ablation: does adding Arts zero-leakage novelty, Kelly SBERT novelty & OpenAlex science lift PR-AUC?
if SECTOR_ONLY:
    _sec_idx = net.index[net["in_sector_v3"]]
    _sec_sets = {
        "sector_meta_t0": CLASSIC + OURS_UNIV,
        "sector_meta + Arts_zero_leakage": CLASSIC + OURS_UNIV + [c for c in ("arts_novelty_rank", "abstract_long_word_share") if c in SECTOR_ONLY],
        "sector_meta + Kelly_SBERT": CLASSIC + OURS_UNIV + [c for c in ("kelly_semantic_novelty_rank",) if c in SECTOR_ONLY],
        "sector_meta + OpenAlex_PaECTER": CLASSIC + OURS_UNIV + [c for c in ("openalex_science_rank", "openalex_fundamental_match") if c in SECTOR_ONLY],
        "sector_ALL_multimodal_t0": CLASSIC + OURS_UNIV + SECTOR_ONLY,
    }
    preds_sec_abl = run_rolling_oot_backtest(
        signals.loc[_sec_idx], net.loc[_sec_idx, "valuable"], net.loc[_sec_idx, "priority_year"],
        net.loc[_sec_idx, "in_sector_v3"], feature_sets=_sec_sets, test_years=TEST_YEARS, lookback=LOOKBACK, random_state=RANDOM_STATE,
    )
    sec_ablation = eval_table(preds_sec_abl, "Humanoid sector ablation", only_sector=True)
    sec_ablation.to_csv(OUT_DIR / "sector_text_science_ablation.csv", index=False)
    print("\n=== Humanoid-Sector-Only Ablation: Arts Lexical vs. Kelly SBERT vs. OpenAlex PaECTER ===")
    display(sec_ablation.round(3))


**Our reading of the robustness check & humanoid text/science ablation.**
- **Secondary economic target (`intl_valuable`):** Achieves PR-AUC `0.993–0.996` (`Lift 3.72×`) primarily because `early_offices_12m` and `has_pct` are already strong structural precursors of multi-office expansion — we treat this transparently as a sanity check rather than an independent discovery.
- **Humanoid-sector multimodal ablation (`n = 330`, `Theory 6` pp. 21–22 & 43):** Adding **OpenAlex + `PaECTER` causal science proximity** produces the single largest sector lift (**PR-AUC `0.299 → 0.347`**, ROC `0.742`, **Lift `3.87×`**). Crucially, **Arts, Hou & Gomez (2021) zero-leakage lexical novelty** (`PR-AUC 0.305`, Lift `3.55×`) outperforms **Kelly et al. pretrained SBERT novelty** (`0.277`, `2.90×`), empirically validating `Theory 6` p. 43's warning about pretrained-encoder drift and topic conflation.


## 5. Interpretability: which signals actually predict breakthroughs? (Theory 6, p. 43)

To ensure our valuation model is not a black box, we report both:
1. **Standardized Logistic Regression coefficients** (exact direction and magnitude on the `t0` proposed feature set);
2. **Holdout Permutation Importance** (`scoring="average_precision"`, PR-AUC drop on the out-of-time 2017 holdout set) for our Gradient Boosting model.


In [ ]:
# Fit on 2012-2016, inspect interpretability on 2017 holdout (zero leakage)
_tr_imp = net["priority_year"].between(2012, 2016) & net["valuable"].notna()
_te_imp = (net["priority_year"] == 2017) & net["valuable"].notna()
_feats_t0 = CLASSIC + OURS_UNIV

_clfs = make_classifiers(random_state=RANDOM_STATE)
_log_m = _clfs["logistic"].fit(signals.loc[_tr_imp, _feats_t0], net.loc[_tr_imp, "valuable"].astype(int))
_hgb_m = _clfs["boosting"].fit(signals.loc[_tr_imp, _feats_t0], net.loc[_tr_imp, "valuable"].astype(int))

# Logistic standardized coefficients (first len(_feats_t0) correspond to the features before missingness indicators)
_coefs = _log_m.named_steps["model"].coef_[0][:len(_feats_t0)]
_perm = permutation_importance(
    _hgb_m, signals.loc[_te_imp, _feats_t0], net.loc[_te_imp, "valuable"].astype(int),
    scoring="average_precision", n_repeats=15, random_state=RANDOM_STATE,
)
importance_df = pd.DataFrame({
    "feature": _feats_t0,
    "group": [SIGNALS[f]["set"] for f in _feats_t0],
    "logistic_std_coef": _coefs,
    "boosting_perm_PR_AUC_drop": _perm.importances_mean,
    "perm_std": _perm.importances_std,
}).sort_values("boosting_perm_PR_AUC_drop", ascending=False)
importance_df.to_csv(OUT_DIR / "feature_importance.csv", index=False)
display(importance_df.round(4))

def _plot_importance():
    top = importance_df.head(14).sort_values("boosting_perm_PR_AUC_drop", ascending=True)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.8, 4.8))
    cols1 = ["#e67e22" if g == "ours" else "#2c7bb6" for g in top["group"]]
    ax1.barh(top["feature"], top["boosting_perm_PR_AUC_drop"], xerr=top["perm_std"], color=cols1, capsize=2)
    ax1.axvline(0, color="black", lw=0.8)
    ax1.set_xlabel("Holdout PR-AUC drop when permuted (2017 test)")
    ax1.set_title("Boosting Permutation Importance (Orange = Ours)")
    ax1.spines[["top", "right"]].set_visible(False)

    top_c = importance_df.reindex(top.index)
    cols2 = ["#27ae60" if c > 0 else "#c0392b" for c in top_c["logistic_std_coef"]]
    ax2.barh(top_c["feature"], top_c["logistic_std_coef"], color=cols2)
    ax2.axvline(0, color="black", lw=0.8)
    ax2.set_yticks([])
    ax2.set_xlabel("Standardized Logistic Regression coefficient (sign)")
    ax2.set_title("Direction of Effect (Green = +, Red = −)")
    ax2.spines[["top", "right"]].set_visible(False)
    plt.suptitle("Section 5 · Feature Importance & Direction (t0 Model)", y=1.02, fontweight="bold")
    plt.tight_layout()
    plt.show()

_plot_importance()
del _plot_importance


**Our reading of feature importance (`Theory 6`, p. 43).** Holdout permutation importance (2017 test set) and standardized Logistic Regression signs agree on a clear, interpretable economic story:
1. **`n_backward_cites`** (`+0.105` PR-AUC drop, coef `+0.726`) and **`scope_cpc4`** (`+0.049`, `+0.353`) — broad, well-grounded inventions that bridge multiple technical subclasses;
2. **`tech_cycle_time`** (`+0.032`, coef **`-0.321`**) — **shorter technology cycle time** (building on fresher prior art) strongly increases breakthrough odds, matching Lai, Wei & Hwang (2026);
3. **`early_offices_12m`** (`+0.024`, `+0.493`) and **`office_US`** (`+0.008`, `+0.433`) — early international filing commitment signals high private valuation by the applicant;
4. **`n_inventors`** (`+0.003`, `+0.097`) and **`examiner_xy_share`** (`+0.001`, **`-0.054`**) — larger R&D teams help, while denser blocking prior art against the filing reduces breakthrough probability.


## 6. Scoring the 2022–2025 humanoid firms (and missing-player robotics extension)

We train the winning `t0` model on the last 5 complete labelled years (`2013–2017`, strictly zero overlap with `2022–2025`), score every recent humanoid family (`2022–2025`), and aggregate to the firm level under **all three Theory 6 (p. 38) lenses**:
1. **Mean predicted probability** (average portfolio quality — favors focused visionaries),
2. **Expected breakthroughs** ($\sum \hat{p}_i$, scale × quality — favors high-volume leaders),
3. **Top-3 mean probability & top-decile share** (breakthrough concentration — key for M&A scouting),
plus our transparent **Rule-Based Future Scorecard** (incorporating OpenAlex `PaECTER` science proximity and Arts zero-leakage novelty on the sector!).


In [ ]:
# Select winning t0 model by out-of-time PR-AUC on the full robotics universe
t0_rows = bt_all[bt_all["feature_set"] == "2_proposed_t0 (classic + ours)"].sort_values("PR_AUC", ascending=False)
BEST_MODEL_NAME = str(t0_rows.iloc[0]["model"])
BEST_FEATS = CLASSIC + OURS_UNIV
print(f"Selected final t0 model: {BEST_MODEL_NAME} (OOT PR-AUC = {t0_rows.iloc[0]['PR_AUC']:.3f}, Lift@10% = {t0_rows.iloc[0]['Lift@10%']:.2f}x)")

final_train_mask = net["priority_year"].between(LAST_LABEL_YEAR - LOOKBACK + 1, LAST_LABEL_YEAR) & net["valuable"].notna()
final_clf = make_classifiers(random_state=RANDOM_STATE)[BEST_MODEL_NAME]
final_clf.fit(signals.loc[final_train_mask, BEST_FEATS], net.loc[final_train_mask, "valuable"].astype(int))

# Score ALL 2022-2025 humanoid sector families
recent_sec_ids = net.index[net["in_sector_v3"] & net["priority_year"].between(*APPLY_YEARS)]
recent_probs = pd.Series(final_clf.predict_proba(signals.loc[recent_sec_ids, BEST_FEATS])[:, 1],
                         index=recent_sec_ids, name="pred_prob")

# Sector-enriched Rule-Based Scorecard (adds OpenAlex PaECTER + Arts zero-leakage + Kelly SBERT on the sector)
_sec_rule_dirs = {k: d["direction"] for k, d in SIGNALS.items() if d["direction"] != 0 and k not in DYNAMIC_2Y}
sec_rule_scores = rule_based_future_score(signals.loc[recent_sec_ids], _sec_rule_dirs, year_series=net.loc[recent_sec_ids, "priority_year"])

# Map each sector family to harmonised corporate firm name (exact same MERGE rules as Step 3 Section 2.2)
fam_app_step0 = load_step0("family_applicants_grouped")
MERGE = {
    "UNITREE": ["HANGZHOU YUSHU TECH"],
    "AGIBOT": ["SHANGHAI ZHIYUAN XINCHUANG TECHNOLOGY"],
    "XIAOMI ROBOT": ["BEIJING XIAOMI ROBOT TECH"],
    "HONDA RESEARCH INSTITUTE EUROPE": ["HONDA RESEARCH INST EUROPE", "HONDA RESEARCH INSTITUTE EUROPE"],
    "ALDEBARAN ROBOTICS": ["ALDEBARAN ROBOTICS", "软银机器人欧洲公司", "奥尔德巴伦机器人公司"],
    "PANASONIC": ["PANASONIC", "MATSUSHITA ELECT IND"],
}
INST_PAT = r"\b(?:UNIV\w*|INST\w*|ACADEM\w*|COLLEGE|SCHOOL|HOSPITAL|LAB\w*|CENT(?:ER|RE)|DAXUE|XUEYUAN)\b"
name_to_firm = {old: new for new, olds in MERGE.items() for old in olds}
fa = fam_app_step0.copy()
fa["firm"] = fa["applicant"].map(name_to_firm).fillna(fa["applicant"])
if "psn_sector" in fa.columns:
    _is_corp = fa["psn_sector"].astype(str).str.upper().str.contains("COMPANY")
else:
    _atype = fa["applicant_type"].astype(str).str.upper() if "applicant_type" in fa.columns else pd.Series("COMPANY", index=fa.index)
    _is_corp = _atype.isin(["COMPANY", "UNKNOWN"])
_is_inst = fa["firm"].astype(str).str.upper().str.contains(INST_PAT, regex=True) & ~fa["firm"].astype(str).str.upper().str.contains("HONDA")
fa["is_firm"] = _is_corp & ~_is_inst
fa_firms = fa[fa["is_firm"]][["docdb_family_id", "firm"]].drop_duplicates()

recent_fam_scores = (fa_firms[fa_firms["docdb_family_id"].isin(recent_sec_ids)]
                     .assign(
                         priority_year=lambda d: d["docdb_family_id"].map(net["priority_year"]),
                         pred_prob=lambda d: d["docdb_family_id"].map(recent_probs),
                         rule_score=lambda d: d["docdb_family_id"].map(sec_rule_scores),
                         has_pct=lambda d: d["docdb_family_id"].map(signals["has_pct"]),
                         has_xy=lambda d: d["docdb_family_id"].map(signals["examiner_has_xy"]),
                         openalex_sci=lambda d: d["docdb_family_id"].map(signals["openalex_science_rank"]) if "openalex_science_rank" in signals.columns else np.nan,
                         arts_nov=lambda d: d["docdb_family_id"].map(signals["arts_novelty_rank"]) if "arts_novelty_rank" in signals.columns else np.nan,
                     ))
recent_fam_scores.to_csv(OUT_DIR / "recent_family_future_scores.csv", index=False)

# Aggregate to firms with >= 5 recent families (or any Step 3 scored firm with >= 1 recent family)
step3_present = pd.read_csv(STEP3_DIR / "present_score.csv") if (STEP3_DIR / "present_score.csv").exists() else pd.DataFrame(columns=["firm", "score"])
step3_firms = set(step3_present["firm"]) if len(step3_present) else set()
firm_counts_recent = recent_fam_scores.groupby("firm")["docdb_family_id"].nunique()
eligible_step4 = sorted(set(firm_counts_recent[firm_counts_recent >= 5].index) | (step3_firms & set(firm_counts_recent.index)))

firm_future = aggregate_firm_future_scores(
    recent_fam_scores[recent_fam_scores["firm"].isin(eligible_step4)],
    firm_col="firm", prob_col="pred_prob", rule_col="rule_score", top_k=3,
)
# Enrich with transparent diagnostic columns
_diag = recent_fam_scores.groupby("firm").agg(
    pct_share_pct=("has_pct", lambda s: s.mean() * 100),
    xy_share_pct=("has_xy", lambda s: s.mean() * 100),
    openalex_sci_rank=("openalex_sci", "mean"),
    arts_novelty_rank=("arts_nov", "mean"),
)
firm_future = firm_future.join(_diag)
firm_future["step3_scored"] = firm_future.index.isin(step3_firms)
firm_future.reset_index().to_csv(OUT_DIR / "future_score.csv", index=False)

print(f"Scored {len(recent_sec_ids):,} recent humanoid families ({APPLY_YEARS[0]}-{APPLY_YEARS[1]}) across {len(firm_future)} firms:")
display(firm_future.head(20).round(3))

# Plot top 15 firms by Future Score
def _plot_future_scores():
    top15 = firm_future.head(15).sort_values("future_score_quality_breakthrough", ascending=True)
    cols = ["#2c7bb6" if s else "#e67e22" for s in top15["step3_scored"]]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.2, 5.2))
    ax1.barh([f.title()[:28] for f in top15.index], top15["future_score_quality_breakthrough"], color=cols)
    for i, (v, n) in enumerate(zip(top15["future_score_quality_breakthrough"], top15["recent_families"])):
        ax1.text(v + 1.0, i, f"{v:.1f} (n={int(n)})", va="center", fontsize=8)
    ax1.set_xlim(0, 118)
    ax1.set_xlabel("Future Score (0–100: quality + top-3 breakthrough + expected hits)")
    ax1.set_title("Future Score (Blue = Step 3 Mature, Orange = Young Entrant)")
    ax1.spines[["top", "right"]].set_visible(False)

    ax2.barh([f.title()[:28] for f in top15.index], top15["mean_prob"], color=cols, alpha=0.85)
    for i, (mp, pct) in enumerate(zip(top15["mean_prob"], top15["pct_share_pct"])):
        ax2.text(mp + 0.008, i, f"{mp:.2f} | PCT {pct:.0f}%", va="center", fontsize=8)
    ax2.set_yticks([])
    ax2.set_xlim(0, max(0.85, float(top15["mean_prob"].max()) + 0.22))
    ax2.set_xlabel("Mean Predicted Breakthrough Probability (2022–2025)")
    ax2.set_title("Average Portfolio Quality & PCT Share")
    ax2.spines[["top", "right"]].set_visible(False)
    plt.suptitle("Section 6 · Step 4 Future Score Ranking (2022–2025 Humanoid Filings)", y=1.02, fontweight="bold")
    plt.tight_layout()
    plt.show()

_plot_future_scores()
del _plot_future_scores


**Our reading of the 2022–2025 firm future scores.** Across **1,712 recent humanoid families** and **42 firms**, the Step 4 ranking clearly distinguishes **global-facing Western visionaries** from **high-volume domestic Chinese waves** and **fading incumbents**:
- **#1 overall: Figure AI (`96.9`)** — combines substantial recent scale (`n = 44`), high average quality (`mean_prob = 0.426`, `18.7` expected breakthroughs), and the highest international commitment among major entrants (`18.2%` PCT).
- **High-volume Chinese wave:** **Guangzhou Ligong (`93.7`, `n = 92`, `36.1` expected hits)**, **Shanghai Fourier (`87.5`, `n = 91`, `33.5` expected hits, `3.3%` PCT)**, **Renxing (`86.1`, `n = 31`)**, and **Shenzhen Zhuji / LimX (`72.6`, `n = 59`)** dominate raw expected breakthroughs, though almost entirely filed domestically in China (`0–3%` PCT).
- **Focused Western quality:** **Apptronik (`91.4`, `n = 5`)** posts the highest average quality (`mean_prob = 0.485`) and `100%` PCT, though on a small portfolio.
- **Only one mature leader stays strong in Step 4:** **UBTECH (`71.9`, `n = 35`, `8.8` expected hits)** continues active filing, whereas **Aldebaran (`54.3`, `n = 3`)** and **Samsung (`23.2`, `n = 2`)** filed almost nothing in `2022–2025`, and **Honda, Toyota, Kawada, and Tmsuk filed zero humanoid families**.


### 6.2 Sensitivity overlay: missing famous players on their 2022–2025 robotics portfolios

Step 3 Section 2.4 showed that **AgiBot (131 robotics families since 2015)**, **Unitree (39)**, and **Boston Dynamics (186)** title most of their humanoid patents as *"dexterous hand"*, *"joint module"*, or *"legged robot"* rather than *"humanoid robot"*. Here we score their `2022–2025` robotics families that appear in our robotics universe (clearly flagged as *"extended robotics portfolio — sensitivity check"*).


In [ ]:
PLAYERS_FILE = STEP3_DIR / "players_in_patstat.csv"
ext_players_scores = pd.DataFrame()
if PLAYERS_FILE.exists():
    pl = pd.read_csv(PLAYERS_FILE)
    pl["player"] = pl["psn_name"].astype(str).str.upper()
    _pat_map = {
        "AGIBOT (robotics ext.)": r"\bAGIBOT\b|ZHIYUAN XINCHUANG",
        "UNITREE (robotics ext.)": r"\bUNITREE\b|HANGZHOU YUSHU",
        "BOSTON DYNAMICS (robotics ext.)": r"\bBOSTON DYNAMICS\b",
        "AGILITY ROBOTICS (robotics ext.)": r"\bAGILITY ROBOTICS\b",
        "TESLA (robotics ext.)": r"\bTESLA\b",
    }
    rows_ext = []
    for label, rx in _pat_map.items():
        fids = set(pl.loc[pl["player"].str.contains(rx, regex=True), "docdb_family_id"].astype(int))
        in_u = sorted(fids & set(net.index[net["priority_year"].between(*APPLY_YEARS)]))
        if not in_u:
            continue
        probs = final_clf.predict_proba(signals.loc[in_u, BEST_FEATS])[:, 1]
        rows_ext.append({
            "player": label,
            "recent_robotics_families_in_universe": len(in_u),
            "mean_prob": float(np.mean(probs)),
            "top3_mean_prob": float(np.mean(np.sort(probs)[::-1][:min(3, len(probs))])),
            "expected_breakthroughs": float(np.sum(probs)),
            "pct_share_pct": float(signals.loc[in_u, "has_pct"].mean() * 100),
        })
    if rows_ext:
        ext_players_scores = pd.DataFrame(rows_ext).set_index("player").sort_values("mean_prob", ascending=False)
        ext_players_scores.to_csv(OUT_DIR / "missing_players_robotics_scores.csv")
        print("Missing players scored on their 2022-2025 robotics families in our universe (sensitivity check):")
        display(ext_players_scores.round(3))
else:
    print("players_in_patstat.csv not found: missing-player robotics extension skipped")


**Our reading of the missing-players robotics sensitivity check.** Scoring the `2022–2025` robotics filings that enter our broader robotics universe (outside strict rule v3) confirms that three prominent players left out by title wording carry strong predicted patent quality: **Boston Dynamics (`mean_prob = 0.654`, `75%` PCT, `n = 4` in sampled universe)**, **Unitree (`0.403`, `66.7%` PCT, `n = 6`)**, and **AgiBot (`0.311`, `n = 19`, `5.9` expected hits, `5.3%` PCT)**. Electrolux should keep **Unitree** and **AgiBot** on its commercial radar alongside the strict-sector shortlist.


## 7. Step 5 Magic Quadrant bridge, sanity checks & deck answer

We merge **Step 3 Present Score (y-axis)**, **Step 4 Future Score (x-axis)**, **Total Families (bubble size)**, **Living Residual Stock**, and **Electrolux Jaffe Proximity (bubble colour)** into `outputs/step4/magic_quadrant_data.csv`, placing young 2023–2025 entrants (`< 10` mature families, no Step 3 score) honestly in a shaded bottom **"Emerging Entrants"** band so they are never distorted by a Volume-only Present Score.


In [ ]:
# Assemble unified Step 5 Magic Quadrant dataset
total_fams_by_firm = fa_firms.groupby("firm")["docdb_family_id"].nunique()
living_path = STEP3_DIR / "living_patents_stock.csv"
prox_path = STEP3_DIR / "electrolux_proximity.csv"

mq_firms = sorted(set(step3_firms) | set(firm_future.head(12).index))
mq = pd.DataFrame(index=mq_firms)
mq.index.name = "firm"
mq["total_families"] = mq.index.map(total_fams_by_firm).fillna(0).astype(int)
mq["present_score"] = mq.index.map(step3_present.set_index("firm")["score"]) if len(step3_present) else np.nan
mq["future_score"] = mq.index.map(firm_future["future_score_quality_breakthrough"])
mq["mean_future_prob"] = mq.index.map(firm_future["mean_prob"])
mq["recent_families"] = mq.index.map(firm_future["recent_families"]).fillna(0).astype(int)
mq["pct_share_pct"] = mq.index.map(firm_future["pct_share_pct"])
if living_path.exists():
    _liv = pd.read_csv(living_path).set_index("firm")
    mq["living_families"] = mq.index.map(_liv["living families"])
    mq["residual_stock"] = mq.index.map(_liv["residual-weighted stock"])
if prox_path.exists():
    _px = pd.read_csv(prox_path).set_index("firm")
    mq["electrolux_proximity"] = mq.index.map(_px["jaffe_proximity"])

mq["band"] = np.where(mq["present_score"].notna(), "Mature (Step 3 + Step 4)", "Emerging Entrant (Step 4 only)")
mq = mq.sort_values(["band", "future_score"], ascending=[False, False])
mq.reset_index().to_csv(OUT_DIR / "magic_quadrant_data.csv", index=False)
display(mq.round(2))

# Preview Step 5 Magic Quadrant (bubble size = total families, colour = Electrolux Jaffe proximity)
def _plot_quadrant():
    df = mq.dropna(subset=["future_score"]).copy()
    # Place Emerging Entrants (no Step 3 Present Score) cleanly in a shaded bottom band at y = 48
    y_plot = df["present_score"].fillna(48.0)
    x_plot = df["future_score"]
    sizes = np.clip(df["total_families"] * 7.0, 60, 750)
    cvals = df["electrolux_proximity"].fillna(df["electrolux_proximity"].median()) if "electrolux_proximity" in df.columns else pd.Series(0.25, index=df.index)

    fig, ax = plt.subplots(figsize=(10.5, 6.8))
    ax.axhspan(42, 54, color="#f2f3f4", alpha=0.85, zorder=0)
    ax.text(2, 43.2, "Emerging Entrants (<10 mature families — Step 3 Present Score N/A; positioned by Step 4 Future Score)",
            fontsize=8, color="#555", style="italic")

    mature_df = df[df["present_score"].notna()]
    x_mid = float(df["future_score"].median())
    y_mid = float(mature_df["present_score"].median()) if len(mature_df) else 75.0
    ax.axvline(x_mid, color="#7f8c8d", ls="--", lw=1.0)
    ax.axhline(y_mid, color="#7f8c8d", ls="--", lw=1.0)

    sc = ax.scatter(x_plot, y_plot, s=sizes, c=cvals, cmap="viridis", alpha=0.85, edgecolor="black", linewidth=0.8, zorder=3)
    for firm, r in df.iterrows():
        y_i = r["present_score"] if pd.notna(r["present_score"]) else 48.0
        lbl = firm.title().replace(" Robotics", "").replace(" Technology", "").replace(" Intelligent", "")[:22]
        ax.annotate(lbl, (r["future_score"], y_i), xytext=(6, 5), textcoords="offset points", fontsize=8)

    cbar = plt.colorbar(sc, ax=ax, pad=0.02)
    cbar.set_label("Jaffe Technological Proximity to Electrolux")
    ax.set_xlim(-5, 108)
    ax.set_ylim(40, 96)
    ax.set_xlabel("Step 4 · Future Score (0–100, Completeness of Vision)")
    ax.set_ylabel("Step 3 · Present Score (RRF, Ability to Execute)")
    ax.set_title("Step 5 Preview · Humanoid Robots Magic Quadrant for Electrolux", fontweight="bold")
    ax.text(98, 93, "LEADERS", ha="right", fontsize=9, fontweight="bold", color="#2c7bb6")
    ax.text(4, 93, "CHALLENGERS", ha="left", fontsize=9, fontweight="bold", color="#7f8c8d")
    ax.text(98, 56, "VISIONARIES", ha="right", fontsize=9, fontweight="bold", color="#e67e22")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    plt.show()

_plot_quadrant()
del _plot_quadrant


In [ ]:
# --- Sanity checks & Deck Answer ---
def check(label, ok):
    print(("OK   - " if ok else "FAIL - ") + label)
    assert ok, f"Check failed: {label}"


check("no 2022-2025 scored family was used in training or testing",
      len(set(recent_sec_ids) & set(net.index[final_train_mask])) == 0)
check("rolling OOT backtest ran across all test years",
      set(preds["test_year"].unique()) == set(TEST_YEARS))
check("proposed t0 model beats random PR-AUC",
      float(t0_rows.iloc[0]["PR_AUC"]) > float(y_test_all.mean()))
check("outputs saved cleanly",
      (OUT_DIR / "future_score.csv").exists() and (OUT_DIR / "magic_quadrant_data.csv").exists())

top_fut = firm_future.head(5)
top_str = ", ".join(f"{f.title()[:22]} ({s:.0f}, n={int(n)})" for f, s, n in zip(top_fut.index, top_fut["future_score_quality_breakthrough"], top_fut["recent_families"]))
best_bench_pr = float(bt_all[bt_all["feature_set"] == "1_benchmark_t0 (classic)"]["PR_AUC"].max())
best_prop_pr = float(t0_rows.iloc[0]["PR_AUC"])
best_dyn_pr = float(bt_all[bt_all["feature_set"] == "3_proposed_t0+2y (+ PageRank/CD)"]["PR_AUC"].max())

answer_box(
    "Which emerging humanoid-robot firms show the strongest future value?",
    f"Top Step 4 Future Score firms (2022-2025): {top_str}.",
    f"Strict out-of-time backtest (2014-2017, base rate {y_test_all.mean():.1%}): "
    f"proposed t0 {BEST_MODEL_NAME} reaches PR-AUC {best_prop_pr:.3f} (Lift@10% {t0_rows.iloc[0]['Lift@10%']:.2f}x) "
    f"vs classic benchmark {best_bench_pr:.3f} and t0+2y dynamic {best_dyn_pr:.3f}.",
)
write_step4_report_index()
